                # Transmon

                This notebook documents the fixed-frequency `Transmon` workflow in `ScQubitsMimic.jl`, focusing on construction, operators, spectrum trends, matrix elements, and wavefunction interpretation.
                

                **Audience:** readers already comfortable with the shared API from notebook 01.

                **Prerequisites:** basic transmon physics (`EJ`, `EC`, and offset charge `ng`).

                **Learning goals:** create a `Transmon`, inspect charge-basis operators, sweep `ng` and `EJ`, and visualize matrix elements and wavefunctions with the Makie extension.
                

                ## Outline

                1. Build a transmon and compare `ω01` against the transmon-limit approximation.
                2. Inspect the exported charge-basis operators.
                3. Sweep `ng` and `EJ` to see charge dispersion and frequency scaling.
                4. Plot wavefunctions and matrix elements using the extension API.
                

In [ ]:
                using ScQubitsMimic
                using CairoMakie
                

In [ ]:
                tmon = Transmon(EJ=30.0, EC=1.2, ng=0.0, ncut=30, truncated_dim=6)
                evals, _ = eigensys(tmon; evals_count=6)
                ω01 = evals[2] - evals[1]
                ω12 = evals[3] - evals[2]
                ω01_approx = sqrt(8 * tmon.EJ * tmon.EC) - tmon.EC

                (
                    EJ_over_EC = round(tmon.EJ / tmon.EC, digits=3),
                    first_six_levels = round.(evals, digits=6),
                    ω01_GHz = round(ω01, digits=6),
                    anharmonicity_GHz = round(ω12 - ω01, digits=6),
                    transmon_limit_estimate_GHz = round(ω01_approx, digits=6),
                    approximation_error_percent = round(abs(ω01 - ω01_approx) / ω01 * 100, digits=3),
                )
                

In [ ]:
                ϕ_vals = range(-π, π; length=5)
                (
                    n_shape = size(n_operator(tmon).data),
                    exp_iϕ_shape = size(exp_i_phi_operator(tmon).data),
                    cosϕ_shape = size(cos_phi_operator(tmon).data),
                    sinϕ_shape = size(sin_phi_operator(tmon).data),
                    potential_samples = round.(potential.(Ref(tmon), ϕ_vals), digits=4),
                )
                

In [ ]:
                ng_vals = collect(range(-0.5, 0.5; length=9))
                ng_spectrum = get_spectrum_vs_paramvals(tmon, :ng, ng_vals; evals_count=3)

                (
                    ng_points = ng_spectrum.param_vals,
                    ω01_vs_ng = round.(ng_spectrum.eigenvalues[:, 2] .- ng_spectrum.eigenvalues[:, 1], digits=6),
                )
                

In [ ]:
                ej_vals = collect(range(10.0, 50.0; length=7))
                ej_spectrum = get_spectrum_vs_paramvals(tmon, :EJ, ej_vals; evals_count=3)

                (
                    EJ_points = ej_spectrum.param_vals,
                    ω01_vs_EJ = round.(ej_spectrum.eigenvalues[:, 2] .- ej_spectrum.eigenvalues[:, 1], digits=6),
                )
                

In [ ]:
                round.(matrixelement_table(tmon, n_operator(tmon); evals_count=4), digits=6)
                

In [ ]:
                plot_wavefunction(tmon, [1, 2, 3]; mode=:abs_sqr)
                

In [ ]:
                plot_matrixelements(tmon, s -> n_operator(s); evals_count=8, mode=:abs)
                

                ## Exercise

                Lower `EJ` to `15.0` GHz while keeping `EC=1.2` GHz. Recompute `ω01` and compare the new anharmonicity against `-EC`.
                

In [ ]:
                tmon_exercise = Transmon(EJ=15.0, EC=1.2, ng=0.0, ncut=30, truncated_dim=4)
                evals_exercise = eigenvals(tmon_exercise; evals_count=3)
                (
                    ω01_GHz = round(evals_exercise[2] - evals_exercise[1], digits=6),
                    anharmonicity_GHz = round((evals_exercise[3] - evals_exercise[2]) - (evals_exercise[2] - evals_exercise[1]), digits=6),
                )
                

                ## Pitfalls And Extensions

                `plot_wavefunction` uses eigenstate indices in Julia's 1-based convention: `1` is the ground state, `2` is the first excited state, and so on.

                The built-in plotting API currently visualizes the charge-basis wavefunction directly. If you need phase-basis reconstructions beyond the exported API, keep that logic notebook-local and label it clearly as a derived analysis rather than a package function.
                

                ## API Covered

                `Transmon`, `n_operator`, `exp_i_phi_operator`, `cos_phi_operator`, `sin_phi_operator`, `potential`, `plot_wavefunction`, and `plot_matrixelements`.
                